This file prepares the DeepPCB dataset for compatibility with YOLO.

In [ ]:
# # Checking for class imbalance
# from pathlib import Path

# root = Path("DeepPCB-master/PCBData")

# results = []

# for subdir in root.iterdir():
#     if subdir.is_dir():
#         for nested in subdir.iterdir():
#             if nested.is_dir():
#                 file_count = sum(1 for f in nested.iterdir() if f.is_file())
#                 results.append((subdir.name, nested.name, file_count))

# # print results
# for parent, child, count in results:
#     print(f"{parent}/{child}: {count}")

group00041/00041: 442
group00041/00041_not: 221
group12000/12000: 28
group12000/12000_not: 14
group12100/12100: 292
group12100/12100_not: 146
group12300/12300: 196
group12300/12300_not: 98
group13000/13000: 432
group13000/13000_not: 216
group20085/20085: 650
group20085/20085_not: 325
group44000/44000: 200
group44000/44000_not: 100
group50600/50600: 158
group50600/50600_not: 79
group77000/77000: 214
group77000/77000_not: 107
group90100/90100: 149
group90100/90100_not: 74
group92000/92000: 240
group92000/92000_not: 120


The dataset seems to be extremely skewed (e.g., 650 images for group 20085 vs. 28 for group 12000). We will handle this imbalance using native YOLO augmentation methods.

In [4]:
import os
import shutil
import random
from collections import defaultdict

# --- Configuration ---
SOURCE_DIR = "DeepPCB-master/PCBData" 
OUTPUT_DIR = "yolo_dataset"
SPLIT_RATIO = 0.8  
IMG_SIZE = (640, 640)

# --- Create YOLO Directory Structure ---
folders = ['images/train', 'images/val', 'labels/train', 'labels/val']
for folder in folders:
    os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

In [5]:
def convert_to_yolo(size, box):
    """Converts absolute bounding box to YOLO normalized format."""
    dw = 1. / size[0]
    dh = 1. / size[1]
    
    x_center = (box[0] + box[2]) / 2.0
    y_center = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    
    x_center = x_center * dw
    w = w * dw
    y_center = y_center * dh
    h = h * dh
    
    return (x_center, y_center, w, h)

# --- Gather and Group Data ---
# Dictionary to hold pairs mapped to their PCB group for stratified splitting
group_dict = defaultdict(list)

# Find all group directories
groups = [d for d in os.listdir(SOURCE_DIR) if d.startswith('group')]

for group in groups:
    group_path = os.path.join(SOURCE_DIR, group)
    group_num = group.replace('group', '') # e.g., '00041'
    
    img_dir = os.path.join(group_path, group_num)
    not_dir = os.path.join(group_path, group_num + '_not')
    
    if not os.path.exists(img_dir) or not os.path.exists(not_dir):
        continue
        
    for img_file in os.listdir(img_dir):
        # YOLO only needs the defective images
        if img_file.endswith('_test.jpg'):
            base_name = img_file.replace('_test.jpg', '')
            
            # Match with notation file
            txt_file = base_name + '.txt'
            txt_path = os.path.join(not_dir, txt_file)
            img_path = os.path.join(img_dir, img_file)
            
            if os.path.exists(txt_path):
                group_dict[group_num].append((img_path, txt_path))

# --- Stratified Split ---
train_pairs = []
val_pairs = []

for group_num, pairs in group_dict.items():
    random.shuffle(pairs)
    split_index = int(len(pairs) * SPLIT_RATIO)
    
    train_pairs.extend(pairs[:split_index])
    val_pairs.extend(pairs[split_index:])

In [6]:
# --- Process and Move Data ---
def process_data(pairs, subset_name):
    for img_path, txt_path in pairs:
        # Standardize output filenames (remove '_test' suffix for cleaner indexing)
        base_name = os.path.basename(img_path).replace('_test.jpg', '')
        out_img_name = base_name + '.jpg'
        out_txt_name = base_name + '.txt'
        
        # 1. Copy Image
        shutil.copy(img_path, os.path.join(OUTPUT_DIR, 'images', subset_name, out_img_name))
        
        # 2. Convert and Save Labels
        out_txt_path = os.path.join(OUTPUT_DIR, 'labels', subset_name, out_txt_name)
        with open(txt_path, 'r') as f_in, open(out_txt_path, 'w') as f_out:
            for line in f_in:
                parts = line.strip().split(' ')
                if len(parts) >= 5:
                    x1, y1, x2, y2 = map(float, parts[:4])
                    class_id = int(parts[4])
                    
                    # 0-index the class ID 
                    yolo_class_id = class_id - 1
                    
                    yolo_bbox = convert_to_yolo(IMG_SIZE, (x1, y1, x2, y2))
                    f_out.write(f"{yolo_class_id} {' '.join(map(str, yolo_bbox))}\n")

print(f"Executing stratified split across {len(group_dict)} PCB groups...")
process_data(train_pairs, 'train')
process_data(val_pairs, 'val')

print(f"Data ready! Train size: {len(train_pairs)} | Val size: {len(val_pairs)}")

Executing stratified split across 11 PCB groups...
Data ready! Train size: 1196 | Val size: 304
